In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

class GaussianNaiveBayesManual:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_features = X.shape[1]

        # Mean and variance for each class & each feature
        self.means_ = {}
        self.vars_ = {}
        self.priors_ = {}

        for c in self.classes_:
            X_c = X[y == c]
            self.means_[c] = X_c.mean(axis=0)
            self.vars_[c] = X_c.var(axis=0) + 1e-9  # small value to avoid /0
            self.priors_[c] = X_c.shape[0] / X.shape[0]

    def _gaussian_log_likelihood(self, X, class_label):
        mean = self.means_[class_label]
        var = self.vars_[class_label]

        # log of Gaussian PDF, feature-wise
        log_prob = -0.5 * np.sum(
            np.log(2.0 * np.pi * var) + ((X - mean) ** 2) / var,
            axis=1
        )
        return log_prob

    def predict(self, X):
        log_posteriors = []

        for c in self.classes_:
            # log P(x|c) + log P(c)
            log_likelihood = self._gaussian_log_likelihood(X, c)
            log_prior = np.log(self.priors_[c])
            log_posteriors.append(log_likelihood + log_prior)

        log_posteriors = np.vstack(log_posteriors).T  # shape: [n_samples, n_classes]
        y_pred = self.classes_[np.argmax(log_posteriors, axis=1)]
        return y_pred


# ----- Use on Iris dataset -----
iris = load_iris()
X = iris.data      # features
y = iris.target    # labels

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train manual Gaussian NB
gnb_manual = GaussianNaiveBayesManual()
gnb_manual.fit(X_train, y_train)
y_pred_manual = gnb_manual.predict(X_test)

print("Manual Gaussian NB accuracy:", accuracy_score(y_test, y_pred_manual))


Manual Gaussian NB accuracy: 0.9111111111111111


In [3]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Load Iris
iris = load_iris()
X = iris.data
y = iris.target

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# In-built GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)

print("Sklearn GaussianNB accuracy:", accuracy_score(y_test, y_pred))


Sklearn GaussianNB accuracy: 0.9111111111111111


Q2..

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 1. Load any dataset – here, Iris
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 2. Define model
knn = KNeighborsClassifier()

# 3. Define hyperparameter grid for K
param_grid = {"n_neighbors": np.arange(1, 21)}

# 4. GridSearchCV to find best K
grid = GridSearchCV(knn, param_grid, cv=5)
grid.fit(X_train, y_train)

print("Best K:", grid.best_params_["n_neighbors"])
print("Best CV score:", grid.best_score_)

# 5. Evaluate on test data
best_knn = grid.best_estimator_
y_pred = best_knn.predict(X_test)
print("Test accuracy with best K:", accuracy_score(y_test, y_pred))
